# T2f Demo Reproduction

This notebook reproduces the T2f workflow on a small ELD subset for a short demo and report.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'original_code'))
sys.path.insert(0, str(ROOT / 'original_code' / 't2f'))

print('Project root:', ROOT)


In [ ]:
from t2f.t2f_base import data_load, t2f_fit

DATASET = 'eld'
LOCAL_LENGTH = 2
GLOBAL_LENGTH = 3
EPOCHS = 1
INPUT_PATH = ROOT / 'datasets' / 'eld' / 'diff_eld.parquet'

train_df, test_df, prepared_df, freq = data_load(
    str(INPUT_PATH),
    'id',
    'timestamp',
    '%Y-%m-%d %H:%M:%S',
    ['tmax', 'tmin', 'tmed'],
    'cost_kwh',
    'load',
    0.2,
    '15min',
    id_frac=0.02,
    len_frac=0.02,
)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('freq:', freq)


In [ ]:
model = t2f_fit(
    train_df=train_df,
    test_df=test_df,
    prepared_df=prepared_df,
    freq=freq,
    series_id_col='id',
    timestamp_col='timestamp',
    covariate_cols=['tmax', 'tmin', 'tmed'],
    weight_col='cost_kwh',
    target_col='load',
    local_length=LOCAL_LENGTH,
    global_length=GLOBAL_LENGTH,
    epochs=EPOCHS,
    model_save_path=str(ROOT / 'results' / 'demo_model' / f'ep{EPOCHS}'),
    hidden_size=32,
    num_layers=1,
    batch_size=8,
)

pred = model.predict(test_df[['tmax', 'tmin', 'tmed']].head(64))
print('prediction shape:', pred.shape)
print(pred.head())


In [ ]:
import json
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

output_dir = ROOT / 'results'
output_dir.mkdir(exist_ok=True)

summary = {
    'dataset': DATASET,
    'freq': freq,
    'epochs': EPOCHS,
    'train_shape': list(train_df.shape),
    'test_shape': list(test_df.shape),
    'prediction_shape': list(pred.shape),
    'sample_prediction': pred.reset_index().head(10).to_dict(orient='records'),
}

with open(output_dir / 'eld_demo_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

plot_df = pred.reset_index().dropna(subset=['timestamp']).copy()
plot_df['timestamp'] = pd.to_datetime(plot_df['timestamp'])
plot_df = plot_df.sort_values('timestamp')

fig, ax = plt.subplots(figsize=(10, 4))
if not plot_df.empty:
    ax.plot(plot_df['timestamp'], plot_df['prediction'], color='royalblue')
    ax.set_title('T2f demo prediction on ELD subset')
else:
    base = test_df.reset_index()[['timestamp', 'load']].head(64).copy()
    base['timestamp'] = pd.to_datetime(base['timestamp'])
    base = base.sort_values('timestamp')
    ax.plot(base['timestamp'], base['load'], color='forestgreen')
    ax.set_title('Observed load on ELD subset')
ax.set_xlabel('timestamp')
ax.set_ylabel('value')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(output_dir / 'eld_demo_plot.png')
print('Saved demo output to:', output_dir)
